# Ejemplo visual: rectificación (PDF pp. 25–27)

Ejecutar en orden. Si falta algo: `pip install plotly ipywidgets`.

In [3]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Puntos del PDF Sesión 1 (pp. 25–26)
src0 = np.array([[140.0, 280.0], [40.0, 140.0], [225.0, 100.0], [350.0, 200.0]])
dst0 = np.array([[50.0, 80.0], [150.0, 80.0], [150.0, 180.0], [50.0, 180.0]])

# H del PDF (p. 27) — referencia
H_pdf = np.array([
    [-0.353, -0.659, 334.276],
    [0.544, -0.112, 115.368],
    [-0.001, 0.004, 1.0],
])

alpha = widgets.FloatSlider(
    value=1.0, min=0.0, max=1.0, step=0.05,
    description='mezcla α',
)
out = widgets.Output()
info = widgets.HTML()


def homografia_4puntos(src, dst):
    A = []
    for (x, y), (u, v) in zip(src, dst):
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    _, _, Vt = np.linalg.svd(np.asarray(A, float))
    H = Vt[-1].reshape(3, 3)
    return H / H[2, 2]


def aplicar_H(H, pts):
    Ph = np.hstack([pts, np.ones((len(pts), 1))])
    Qh = (H @ Ph.T).T
    return Qh[:, :2] / Qh[:, 2:3]


def actualizar(*_):
    H = homografia_4puntos(src0, dst0)
    src_m = aplicar_H(H, src0)
    # interpolar visualmente borde origen → borde destino
    a = alpha.value
    mid = (1 - a) * src0 + a * src_m

    info.value = (
        f"<b>PDF:</b> esquinas foto → cuadrado destino; α=0 foto, α=1 rectificado<br>"
        f"H estimada:<br>"
        f"[{H[0,0]:.3f} {H[0,1]:.3f} {H[0,2]:.3f}]<br>"
        f"[{H[1,0]:.3f} {H[1,1]:.3f} {H[1,2]:.3f}]<br>"
        f"[{H[2,0]:.3f} {H[2,1]:.3f} {H[2,2]:.3f}]<br>"
        f"H del PDF (p.27): [{-0.353} {-0.659} {334.276}] … (misma idea; puede diferir en escala/orden de esquinas)"
    )

    def poly(pts):
        return np.vstack([pts, pts[0]])

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=poly(src0)[:, 0], y=poly(src0)[:, 1], mode='lines+markers+text',
        text=['1', '2', '3', '4', ''], textposition='top center',
        line=dict(color='#1f77b4', width=3), marker=dict(size=10),
        name='foto (PDF p.25)',
    ))
    fig.add_trace(go.Scatter(
        x=poly(dst0)[:, 0], y=poly(dst0)[:, 1], mode='lines+markers+text',
        text=["1'", "2'", "3'", "4'", ''], textposition='bottom center',
        line=dict(color='#d62728', width=3), marker=dict(size=10),
        name='destino (PDF p.26)',
    ))
    fig.add_trace(go.Scatter(
        x=poly(mid)[:, 0], y=poly(mid)[:, 1], mode='lines+markers',
        line=dict(color='#2ca02c', width=4), marker=dict(size=8),
        name=f'mezcla α={a:.2f}',
    ))
    # líneas de correspondencia
    for i in range(4):
        fig.add_trace(go.Scatter(
            x=[src0[i, 0], dst0[i, 0]], y=[src0[i, 1], dst0[i, 1]],
            mode='lines', line=dict(color='gray', width=1, dash='dot'),
            showlegend=False, hoverinfo='skip',
        ))

    fig.update_layout(
        width=640, height=560,
        xaxis=dict(range=[0, 400], scaleanchor='y', scaleratio=1, title='x (px)'),
        yaxis=dict(range=[300, 50], title='y (px)'),  # origen arriba como imagen
        margin=dict(l=40, r=20, t=40, b=40),
        legend=dict(orientation='h', y=1.1),
        title=dict(text='Rectificación: de cuadrilátero (foto) a cuadrado', x=0.5),
    )

    with out:
        out.clear_output(wait=True)
        fig.show()


alpha.observe(actualizar, names='value')
ui = widgets.VBox([alpha, info, out])
display(ui)
actualizar()